# M06-01 — Explain y DAG

[← Anterior](01-teoria.ipynb) · [Siguiente →](03-lab-cache-particionado.ipynb)

Este fichero es el **guion**. No lo rellenes aquí: **crea tu propio notebook** y ve construyéndolo celda a celda.

## Qué vas a hacer

Demostrar que cinco transformaciones no lanzan job, y señalar scan + filtro en el plan formateado.

## 0 — Crea tu notebook

1. En el explorador, abre la carpeta `notebooks/trabajo/`.
2. Clic derecho → **New File…**
3. Nombre exacto: `M06-01-explain-dag.ipynb` (incluye `.ipynb`).
4. Ábrelo. Arriba a la derecha (o `F1` → `Notebook: Select Notebook Kernel`) elige **Python (NovaShop)**.
5. Deja **este** guion a un lado (pestaña) y escribe **solo** en el tuyo.

## Cómo organizar *tu* notebook (siempre)

En cada paso creas **dos celdas**, en este orden:

1. **Markdown** — qué vas a hacer y por qué, con tus palabras.
2. **Código** — el de la celda de código del paso. Lo ejecutas (`Shift+Enter`), miras la salida y, si no cuadra, lo mejoras.

No dejes un muro de código sin explicación. Un notebook se lee de arriba abajo, como un cuaderno.

> Kernel **Python (NovaShop)**. Si no aparece: terminal → `bash .devcontainer/setup.sh` → vuelve a elegir kernel.


## Spark UI

Abre la pestaña **Ports** del Codespace → puerto **4040**. Anota el último Job Id que ves *antes* del paso 2 (puede ser 0).


### Paso 1 — Plan sin ejecutar

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Imprimir el objeto DataFrame no dispara jobs. Encadeno wheres y un select.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** El Job Id **más alto** de Spark UI **no cambia** al ejecutar esta celda.

**Por qué este paso.** Lazy de verdad: no hay acción.


In [ ]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


from pyspark.sql.functions import col

spark = get_spark("novashop-m06")
planned = (
    spark.read.parquet(str(STAGING / "fact_lines"))
    .where(col("is_billable"))
    .where(col("gmv_line") > 0)
    .where(col("channel_norm").isin("web", "app"))
    .select("order_id", "customer_id", "gmv_line", "order_month")
)
print(planned)


### Paso 2 — Una acción, un DAG

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

count obliga a recorrer las particiones. Miro UI: un job nuevo.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** Un job nuevo. En Jobs, el DAG muestra al menos un stage. El count es líneas cobrables web/app con GMV > 0 (varios cientos).

**Por qué este paso.** Si cada celda crea un job, tienes un `.show()` de debug por medio.


In [ ]:
print(planned.count())


### Paso 3 — Lee el plan

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Busco FileScan parquet (o Scan) y Filter. No traduzco cada operador Catalyst.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** Aparece el path `fact_lines` y predicados `is_billable` / `gmv_line` / `channel_norm`.

**Por qué este paso.** Copia en Markdown las dos líneas que identifican scan y filtro.


In [ ]:
planned.explain("formatted")


## Comprueba

Antes de dar el lab por cerrado, vuelve a ejecutar de arriba abajo (**Run All**) y verifica:

Encadena un `.where(...)` extra **sin** `count` y mira Jobs.
No hay job nuevo. Escríbelo.


## Mejora — explain(True) vs formatted

Compara `explain(True)` con `explain("formatted")`. ¿Dónde se ve el filtro empujado al scan?

Si te atasca, el código está en la celda siguiente.


In [ ]:
En el físico / formatted, `PushedFilters` o el Filter junto al FileScan indica predicate pushdown. Si el filtro no aparece, lo aplicaste *después* de un select que ya tiró la columna.


## Si algo falla

| Qué ves | Suele ser | Qué haces |
|---------|-----------|-----------|
| UI vacía / 404 | Puerto 4040 no reenviado | Ports del Codespace → 4040 |
| Cada celda crea un job | Tienes un show de debug | Coméntalo mientras mides |
| Dos sesiones | SparkSession() extra | Solo get_spark() |


## Siguiente

Cuando hayas **comprobado** y (si quieres) **mejorado**, abre [M06-02 cache](03-lab-cache-particionado.ipynb).
